In [0]:
# Product Category Mapping Table

# Import necessary Spark functions for transformations
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# Load silver-layer products table
# Load bronze-layer product category translation table

df_products = spark.table("retail_catalog.silver.products")
df_translation = spark.table("retail_catalog.bronze.product_category_name_translation")

In [0]:
# Join products with translation table to get English category names
# Map fine-grained English category names into broader grouped categories for analytics

df_products_enriched = (
    df_products
    .join(
        df_translation,
        on="product_category_name",
        how="left"
    )
)

products = (
    df_products_enriched
    .withColumn(
        "grouped_category",
        F.when(col("product_category_name_english").isin(
            "bed_bath_table","furniture_decor","housewares","garden_tools",
            "kitchen_dining_laundry_garden_furniture","office_furniture",
            "home_appliances","home_appliances_2","home_confort",
            "home_comfort_2","furniture_mattress_and_upholstery",
            "furniture_living_room","furniture_bedroom","flowers","la_cuisine"
        ), "Home & Furniture")

        .when(col("product_category_name_english").isin(
            "health_beauty","perfumery","diapers_and_hygiene"
        ), "Health & Beauty")

        .when(col("product_category_name_english").isin(
            "computers_accessories","computers","tablets_printing_image",
            "telephony","fixed_telephony","electronics","consoles_games",
            "audio","air_conditioning","cine_photo",
            "small_appliances","small_appliances_home_oven_and_coffee"
        ), "Electronics & Technology")

        .when(col("product_category_name_english").isin(
            "fashion_bags_accessories","fashion_shoes","fashion_male_clothing",
            "fashio_female_clothing","fashion_underwear_beach",
            "fashion_sport","fashion_childrens_clothes",
            "luggage_accessories","watches_gifts"
        ), "Fashion & Accessories")

        .when(col("product_category_name_english") == "sports_leisure",
              "Sports & Leisure")

        .when(col("product_category_name_english").isin(
            "baby","toys"
        ), "Kids & Baby")

        .when(col("product_category_name_english").isin(
            "food","food_drink","drinks"
        ), "Food & Beverages")

        .when(col("product_category_name_english").isin(
            "books_technical","books_general_interest","books_imported",
            "music","cds_dvds_musicals","dvds_blu_ray",
            "musical_instruments","art","arts_and_craftmanship",
            "party_supplies","christmas_supplies"
        ), "Books, Media & Entertainment")

        .when(col("product_category_name_english").isin(
            "auto","construction_tools_construction",
            "costruction_tools_garden","costruction_tools_tools",
            "construction_tools_lights","construction_tools_safety",
            "signaling_and_security"
        ), "Automotive & Tools")

        .otherwise("Other / Business & Misc")
    )
)

# Select columns for output reference table including mapped category
products = products.select("product_id","product_weight_g","product_length_cm","product_height_cm","product_width_cm","product_category_name","product_category_name_english","grouped_category")

In [0]:
# Show enriched product reference table for analysis or export
products

DataFrame[product_id: string, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, product_category_name: string, product_category_name_english: string, grouped_category: string]